In [ ]:
# import pandas --- spreadsheet library
import pandas as pd
import numpy as np
import os

# import matplotlib --- plotting library
import matplotlib.pyplot as plt

from plotting import make_overview, make_yield_plot, compare, smoothen
from myio import read_in_weather_data

In [ ]:
import sys
sys.path.append('../PalmSim')
sys.path.append('..')
from palmsim import PalmField

In [ ]:
%matplotlib inline

In [ ]:
def read_in_observed_yield(party, fn):
    """ Read in a csv of observed yields and return the df. """
    
    fdir = '../../Data/Calibration Data/{:}/Clean data'.format(party)

    ydir = 'Yield'

    fp = os.path.join(fdir,ydir,fn)

    # observed yield
    dfo = pd.read_csv(fp)
    dfo = dfo.set_index(pd.to_datetime(dfo['Month']))
    dfo = dfo.dropna(how='all')
    dfo['Y (kg/ha/yr)'] = 1000*dfo['Y (t/ha/mo)']*(365/dfo.index.days_in_month)
    
    return dfo

# a small test to check the function
assert read_in_observed_yield('Sinarmas', 'Hnae-1986.csv') is not None

In [ ]:
def fetch_dfs(fdirs, fname):
    """ Read in spreadsheets. """
    
    dfs = {} 

    which = 'observed'
    fpath = os.path.join(fdirs[which], fname)
    dfs[which] = pd.read_csv(fpath, index_col = 'Month')
        
    which = 'potential'
    fpath = os.path.join(fdirs[which], fname)
    dfs[which] = pd.read_csv(fpath, index_col = 'date')
    
    which = 'water-limited'
    fpath = os.path.join(fdirs[which], fname)
    dfs[which] = pd.read_csv(fpath, index_col = 'date')

    for key, df in dfs.items():
        
        df.index = pd.to_datetime(df.index)
        dfs[key] = df
    
    return dfs

In [ ]:
fdirs = ['observed', 'potential', 'water-limited']

fdirs = {k:os.path.join("results", k) for k in fdirs}

In [ ]:
fdirs

In [ ]:
fnames = os.listdir(fdirs['observed'])

fname = fnames[0]

In [ ]:
plt.rc('font', family='serif')
# plt.rc('xtick', labelsize='x-small')
# plt.rc('ytick', labelsize='x-small')

In [ ]:
overview = pd.read_excel('summary_table.xls')

In [ ]:
parties = overview['Source'].unique()

In [ ]:
nparties = len(parties)

In [ ]:
cmap = plt.cm.get_cmap('Accent')

colors = {x: cmap(i/nparties) for i,x in enumerate(parties)}


In [ ]:
colors

In [ ]:
import datetime

## Special plot suggested by PG

new files -> outliers?!

In [ ]:
only_one = False

fig, ax = plt.subplots()

for i, fname in enumerate(fnames):
    
    info = {k:v for k,v in zip(list(overview), overview.loc[row].values[0])}
    
    YOP = info['Year of planting']
    
    party = info['Source']

    dfs = fetch_dfs(fdirs, fname)

    dfo = dfs['observed']
    dfyw = dfs['water-limited']
    dfyp = dfs['potential']
    
    Yo = 0.001*dfo['Y (kg/ha/yr)']
    Yyw = 0.001*dfyw['generative_FFB_production (t/ha/yr)']
    Yyp = 0.001*dfyp['generative_FFB_production (t/ha/yr)']
    
    ix_mid = Yo.index[int(len(Yo)/2)]
    
    YAP = ix_mid.year + (1/365)*ix_mid.dayofyear - YOP
    
    row = overview['Filename_clean'] == fname
    
    window = 12
    
    Yo_avg = Yo.resample('1M').mean().rolling(window, center=True).mean()
    Yyw_avg = Yyw.resample('1M').interpolate().rolling(window, center=True).mean()
    Yyp_avg = Yyp.resample('1M').interpolate().rolling(window, center=True).mean()
    
    gap_yw = Yo_avg/Yyw_avg
    gap_yp = Yo_avg/Yyp_avg
    
    yws = gap_yw.loc[gap_yw.index.month==1]
    ywxs = yws.index.year - YOP
    ywys = yws.values
    
    mask = ywys < 1.3
    ywxs = ywxs[mask]
    ywys = ywys[mask]
    
    ax.scatter(ywxs, ywys, marker='o', color=colors[party]) 
    
ax.axhline(1, color='black', ls='dashed')    
ax.set_ylim(0,2)
ax.set_xlim(0, 25)

print(i)

In [ ]:
only_one = False

for fname in fnames:

    dfs = fetch_dfs(fdirs, fname)

    overview = pd.read_excel('summary_table.xls')

    row = overview['Filename_clean'] == fname

    info = {k:v for k,v in zip(list(overview), overview.loc[row].values[0])}

    title = info['Filename_weatherdata_clean'].split('.')[0]

    filepath = os.path.join("plots","incl_yield_factors","{:}.png".format(title))

    make_overview(dfs['water-limited'],
                  dfs['potential'],
                  dfs['observed'],
                  filepath, window=12, title=title)
    
    filepath = os.path.join("plots","yield_only","{:}.png".format(title))
    
    make_yield_plot(dfs['water-limited'],
                  dfs['potential'],
                  dfs['observed'],
                  filepath, window=12, title=title)    
        
    if only_one:
        break;
        
    plt.close()
    plt.close()
    

## Illustration that maw is superior to plain totals placed at 31 december for aggregation

In [ ]:
df = dfs['observed']

s = df['Y (kg/ha/yr)']

In [ ]:
import datetime
def mytotal(s):
    return pd.Series({datetime.datetime(year=year, month=12, day=31):s.loc[s.index.year == year].mean()
           for year in s.index.year.unique()})

In [ ]:
s0 = s
s1 = s.rolling(12,center=True).mean()
s2 = s.rolling(12,center=True,win_type='triang').mean()
s3 = mytotal(s)

s2s = s2[s2.index.month == 12]

ax = s0['2008':'2011'].plot(label='original')
s1['2008':'2011'].plot(label='maw square', color='red')
s2['2008':'2011'].plot(label='maw tri', color='blue')
ax.scatter(s3.index, s3.values, color='red', label='total @ Dec 31st')

ax.scatter(s2s.index, s2s.values, color='blue', label='maw @ Dec 31st')

ax.legend()